# Preprocessing Notebook

This notebook prepares analysis-ready tables from the raw YouTube collection outputs.

**How to use:**
- Update `PERSON` below to switch datasets (e.g., `bodeoum_tv` or `mimiminu`).
- Rerun all cells to produce outputs under `02-preprocessing/outputs/<person>/`.


## 0) Imports and configuration
This cell sets the dataset name and all input/output paths.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm

# ---- CONFIG ----
PERSON = "bodeoum_tv"  # change to 'mimiminu' when needed
LANG = "ko"

REPO_ROOT = Path.cwd().resolve()
DATA_ROOT = REPO_ROOT / "data"
INPUT_CSV = DATA_ROOT / "input" / f"{PERSON}.csv"
YOUTUBE_JSONL = DATA_ROOT / PERSON / f"{PERSON}_youtube.jsonl"
COMMENTS_DIR = DATA_ROOT / PERSON / "comments"
TRANSCRIPTS_DIR = DATA_ROOT / PERSON / "transcripts"

OUTPUT_DIR = REPO_ROOT / "02-preprocessing" / "outputs" / PERSON
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using PERSON=", PERSON)
print("Input CSV:", INPUT_CSV)
print("YouTube JSONL:", YOUTUBE_JSONL)
print("Comments dir:", COMMENTS_DIR)
print("Transcripts dir:", TRANSCRIPTS_DIR)
print("Output dir:", OUTPUT_DIR)


## 1) Load window map (CSV)
This builds a `slot -> video_url -> video_id` mapping based on the CSV order.


In [ ]:
def extract_video_id(url: str) -> str | None:
    if "watch?v=" in url:
        return url.split("watch?v=", 1)[1].split("&", 1)[0]
    if "youtu.be/" in url:
        return url.split("youtu.be/", 1)[1].split("?", 1)[0]
    return None

window_df = pd.read_csv(INPUT_CSV)
window_df["video_id"] = window_df["video"].astype(str).apply(extract_video_id)

slot_rows = []
for window_label in ("pre", "cancel", "post"):
    rows = window_df[window_df["window"].str.lower() == window_label]
    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        slot = f"{window_label}-{i}"
        slot_rows.append({
            "slot": slot,
            "window": window_label,
            "video": row["video"],
            "video_id": row["video_id"],
        })

slot_df = pd.DataFrame(slot_rows)
slot_df.head()


## 2) Extract video metadata (`videos.csv`)
Parses `<person>_youtube.jsonl` and keeps records where `record_type == "video"`.


In [ ]:
video_rows = []
with open(YOUTUBE_JSONL, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record.get("record_type") != "video":
            continue
        video_rows.append({
            "video_id": record.get("video_id"),
            "title": record.get("title"),
            "video_url": record.get("video_url"),
            "create_time": record.get("create_time"),
            "view_count": record.get("view_count"),
            "like_count": record.get("like_count"),
            "comment_count": record.get("comment_count"),
            "window_label": record.get("window_label"),
        })

videos_df = pd.DataFrame(video_rows)
videos_df.to_csv(OUTPUT_DIR / "videos.csv", index=False)
videos_df.head()


## 3) Flatten comments (`comments.csv`)
Reads `comments/*.info.json` and extracts comment fields + window slot.


In [ ]:
comment_rows = []
comment_files = sorted(COMMENTS_DIR.glob("*.info.json"))

for path in tqdm(comment_files, desc="comments"):
    slot = path.stem.replace(".info", "")
    window = slot.split("-")[0]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    video_id = payload.get("id")
    comments = payload.get("comments", [])
    for c in comments:
        comment_rows.append({
            "comment_id": c.get("id"),
            "parent_id": c.get("parent"),
            "author": c.get("author"),
            "author_id": c.get("author_id"),
            "timestamp": c.get("timestamp"),
            "like_count": c.get("like_count"),
            "text": c.get("text"),
            "video_id": video_id,
            "window": window,
            "slot": slot,
        })

comments_df = pd.DataFrame(comment_rows)
comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
comments_df.head()


## 4) Build reply edges (`edges.csv`)
Creates reply graph edges from `comments.csv`.


In [ ]:
edges_df = comments_df[comments_df["parent_id"].notna() & (comments_df["parent_id"] != "root")].copy()
edges_df = edges_df.rename(columns={
    "comment_id": "source_comment_id",
    "parent_id": "target_comment_id",
})
edges_df = edges_df[[
    "source_comment_id",
    "target_comment_id",
    "author_id",
    "video_id",
    "window",
    "slot",
]]
edges_df.to_csv(OUTPUT_DIR / "edges.csv", index=False)
edges_df.head()


## 5) Parse transcripts (`transcripts.csv`)
Reads `transcripts/*.ko.txt` and attaches window/slot.


In [ ]:
transcript_rows = []
for path in sorted(TRANSCRIPTS_DIR.glob(f"*.{LANG}.txt")):
    name = path.name
    # expected: pre-1.ko.txt / cancel-1.ko.txt / post-1.ko.txt
    slot = name.split(".")[0]
    window = slot.split("-")[0]
    text = path.read_text(encoding="utf-8")
    transcript_rows.append({
        "slot": slot,
        "window": window,
        "transcript_text": text,
    })

transcripts_df = pd.DataFrame(transcript_rows)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)
transcripts_df.head()


## 6) Text cleaning
Creates `text_clean` for comments and transcripts without overwriting raw text.


In [ ]:
import re

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = url_re.sub("", text)
    text = whitespace_re.sub(" ", text).strip()
    return text

comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

comments_df[["text", "text_clean"]].head()


## 7) Summary
Quick sanity checks on row counts and output paths.


In [ ]:
print("videos.csv", len(videos_df))
print("comments.csv", len(comments_df))
print("edges.csv", len(edges_df))
print("transcripts.csv", len(transcripts_df))
print("Output dir:", OUTPUT_DIR)
